In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path

OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

## DE1 - Load and profile

Load both sheets, check the column names and types, and report row counts and missing values
per column.

In [2]:
# Read in the data and combine them into one dataframe

df = pd.read_excel("data/raw/online_retail_II.xlsx", sheet_name=None)
df_merged = pd.concat([df['Year 2009-2010'], df['Year 2010-2011']], ignore_index=True)

df_merged.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df_merged.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Data augmentation: splitting StockCode

First we split StockCode into three features for exploration. ProductCode will be the actual product identifier, Variation is the subtype given by a single character (color, style), and NonProductFlag contains abnormal items (postage, bank charges).

The ProductCode pattern is anchored at both ends, because a real stock code is digits followed by optional letters and nothing else. An unanchored `(\d+)` would pull the first run of digits out of any string and quietly turn non-products into products: C2 (CARRIAGE) and ADJUST2 would both become product 2, TEST001 would become product 001, SP1002 would collide with the real product 1002, and the gift vouchers gift_0001_10 to gift_0001_90 would all collapse into a single product 0001.

In [4]:
# Split the stockcode into two features
df_split = df_merged.copy()
df_split['ProductCode'] = df_split['StockCode'].astype(str).str.extract(r'^(\d+)\D*$', expand=False)
df_split['Variation'] = df_split['StockCode'].astype(str).str.extract(r'^\d+(\D*)$', expand=False)
df_split['NonProductFlag'] = df_split['StockCode'].where(df_split['ProductCode'].isna())
df_split.drop(['StockCode'], axis=1, inplace=True)

In [5]:
# Show the stock code an description for all unique variations
df_split.loc[df_split['Variation'].notna(), ['ProductCode', 'Variation', 'Description']].drop_duplicates()

,ProductCode,Variation,Description
0,85048,,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323,P,PINK CHERRY LIGHTS
2,79323,W,WHITE CHERRY LIGHTS
3,22041,,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,,STRAWBERRY CERAMIC TRINKET BOX
...,...,...,...
1062371,23090,,missing
1063082,85123,A,CREAM HANGING HEART T-LIGHT HOLDER
1064015,85175,,NaN
1064386,23169,,smashed


We can look at the description for each NonProductFlag to confirm they are not actually purchases. After this, we will drop any item with no product code.

In [6]:
df_split.loc[df_split['NonProductFlag'].notna(), ['ProductCode', 'NonProductFlag', 'Description']].drop_duplicates()

,ProductCode,NonProductFlag,Description
89,NaN,POST,POSTAGE
735,NaN,D,Discount
2377,NaN,DCGS0058,MISO PRETTY GUM
2378,NaN,DCGS0068,DOGS NIGHT COLLAR
2379,NaN,DOT,DOTCOM POSTAGE
...,...,...,...
804716,NaN,DCGS0069,ebay
804717,NaN,DCGS0068,ebay
804718,NaN,DCGS0067,ebay
804719,NaN,DCGS0003,ebay


### Row counts and missing values per column

In [7]:
# Print missing values per column
print(df_split.isnull().sum())
print()
print("Rows with quantity = 0: ", (df_split['Quantity'] == 0).sum())
print("Rows with quantity < 0: ", (df_split['Quantity'] < 0).sum())
print()
print("Duplicated rows: ", df_split.duplicated().sum())

Invoice                 0
Description          4382
Quantity                0
InvoiceDate             0
Price                   0
Customer ID        243007
Country                 0
ProductCode          6093
Variation            6093
NonProductFlag    1061278
dtype: int64

Rows with quantity = 0:  0
Rows with quantity < 0:  22950



Duplicated rows:  34335


## DE2 - Cleaning: dropping unneeded rows

- Anything with no product code is not a product, and should be removed.

- Anything with no customer ID cannot be used for the analysis, and should be removed.

- Rows with a price of zero or less are not sales (give-aways, write-offs, data errors), and should be removed.

- Rows with negative quantities are left, we will later take the net of purchases and returns for each customer/product so leaving these should account for them. For the same reason, rows with invoices starting with C (representing cancellations) are not removed. Removing them here instead would leave the original purchase standing on its own.

- Duplicated rows are removed. Some of these are due to overlap in the two sheets provided, some are likely due to input mistakes.

In [8]:
df_reduced = df_split.copy()

print(f"Initial number of rows: {df_reduced.shape[0]:,}")

# Remove anything without a product id
df_reduced = df_reduced.loc[df_reduced['ProductCode'].notna()]
print(f"Rows remaining after requiring product IDs: {df_reduced.shape[0]:,}")

# Remove anything without a customer id
df_reduced = df_reduced.loc[df_reduced['Customer ID'].notna()]
print(f"Rows remaining after requiring customer IDs: {df_reduced.shape[0]:,}")

# Remove non-sales
df_reduced = df_reduced.loc[df_reduced['Price'] > 0]
print(f"Rows remaining after removing price <= 0: {df_reduced.shape[0]:,}")

# Remove duplicated rows
df_reduced = df_reduced.drop_duplicates()
print(f"Rows remaining after removing duplicates: {df_reduced.shape[0]:,}")

df_reduced.head()

Initial number of rows: 1,067,371
Rows remaining after requiring product IDs: 1,061,278
Rows remaining after requiring customer IDs: 820,626
Rows remaining after removing price <= 0: 820,565


Rows remaining after removing duplicates: 794,163


,Invoice,Description,Quantity,InvoiceDate,Price,Customer ID,Country,ProductCode,Variation,NonProductFlag
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,85048,,NaN
1,489434,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,79323,P,NaN
2,489434,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,79323,W,NaN
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,22041,,NaN
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,21232,,NaN


In [9]:
# Make sure there are no cancellations with positive quantities (should be false)
print(((df_reduced['Invoice'].str.contains('C') == True) & (df_reduced['Quantity'] > 0)).any())

False


## DE2 - Cleaning: dropping unneeded features

- Description
    - Redundant, we can use product code to identify the product
- NonProductFlag
    - Generated solely to determine the meaning of the stockcode, not needed for analysis.
- Price, Country
    - Not relevant to the analysis

ML6 asks for the segments to be described by their top products and countries, so we save a name for each product and a country for each customer before dropping those columns.

In [10]:
# Lookups for ML6, saved before the columns are dropped
product_names = (df_reduced.dropna(subset=['Description'])
                 .groupby('ProductCode')['Description'].agg(lambda s: s.mode().iat[0]))
customer_country = df_reduced.groupby('Customer ID')['Country'].agg(lambda s: s.mode().iat[0])

df_sliced = df_reduced.drop(columns=['Description', 'Price', 'Country', 'NonProductFlag'])

# Removing the full df from memory
# del df, df_merged

print(df_sliced.head())

  Invoice  Quantity         InvoiceDate  Customer ID ProductCode Variation
0  489434        12 2009-12-01 07:45:00      13085.0       85048          
1  489434        12 2009-12-01 07:45:00      13085.0       79323         P
2  489434        12 2009-12-01 07:45:00      13085.0       79323         W
3  489434        48 2009-12-01 07:45:00      13085.0       22041          
4  489434        24 2009-12-01 07:45:00      13085.0       21232          


## R4 - Redacting Customer IDs

We were asked to redact all customer IDs. We can do that by replacing each with a random new id.

In [11]:
rng = np.random.default_rng(42)

df_obscured = df_sliced.copy()
ids = df_obscured['Customer ID'].dropna().unique()
shuffled = rng.permutation(ids)

mapping = {cid: i for i, cid in enumerate(shuffled)}
df_obscured['AnonID'] = df_obscured['Customer ID'].map(mapping).astype('Int64')
customer_country.index = customer_country.index.map(mapping)

df_obscured = df_obscured.drop(columns='Customer ID')

df_obscured.head()

,Invoice,Quantity,InvoiceDate,ProductCode,Variation,AnonID
0,489434,12,2009-12-01 07:45:00,85048,,4393
1,489434,12,2009-12-01 07:45:00,79323,P,4393
2,489434,12,2009-12-01 07:45:00,79323,W,4393
3,489434,48,2009-12-01 07:45:00,22041,,4393
4,489434,24,2009-12-01 07:45:00,21232,,4393


## DE3 - Extracting the test section

All data from the most recent two months is removed. This will be used to test the results of the recommendation engine.

In [12]:
# get rows within the last two months

latestDateTime = df_obscured['InvoiceDate'].max()
cutoff = latestDateTime - pd.DateOffset(months=2)

train = df_obscured[df_obscured['InvoiceDate'] < cutoff]
test = df_obscured[df_obscured['InvoiceDate'] >= cutoff]

print(f"Train set size: {train.shape[0]:,}")
print(f"Test set size: {test.shape[0]:,}")

Train set size: 675,720
Test set size: 118,443


A time split on its own is not enough. R2 asks us to hold out, for each test customer, purchases of products they never bought before. A customer reordering something they already buy every month is not a test of a recommender: that pair is already in the training matrix, so a hit would be scored for something the model was handed.

Customers who only appear in the test window are dropped as well, because there is no history to recommend from.

In [13]:
# R2: hold out only the products each test customer had never bought before
seen = set(map(tuple, train[['AnonID', 'ProductCode']].drop_duplicates().to_numpy()))
known_customers = set(train['AnonID'].unique())

pairs = test[['AnonID', 'ProductCode']].drop_duplicates()
n_all = len(pairs)

pairs = pairs[pairs['AnonID'].isin(known_customers)]
n_cold = n_all - len(pairs)

pairs = pairs[[tuple(p) not in seen for p in pairs.to_numpy()]]

print(f"Test customer/product pairs: {n_all:,}")
print(f"  dropped, no training history: {n_cold:,}")
print(f"  dropped, repeat purchase: {n_all - n_cold - len(pairs):,}")
print(f"  held out as genuinely new: {len(pairs):,}")

Test customer/product pairs: 93,928
  dropped, no training history: 13,554
  dropped, repeat purchase: 31,907
  held out as genuinely new: 48,467


## DE4 - Transforming into the sparse matrix

In [14]:
print("Unique Customers: ", train['AnonID'].unique().shape[0])
print("Unique Products: ", train['ProductCode'].unique().shape[0])
print("Unique Products (with variations): ", train['ProductCode'].str.cat([train['Variation']], sep=',').unique().shape[0])

Unique Customers:  5497
Unique Products:  3785
Unique Products (with variations):  4543


Including the product variations would add about 760 extra products, which would make the recommendations less accurate. They would also be less useful, since it would very often recommend different variations of an item the person already bought.

In [15]:
train_simple = train.drop(columns=['Variation', 'InvoiceDate'])
test_simple = test.drop(columns=['Variation', 'InvoiceDate'])

train_simple.head()

,Invoice,Quantity,ProductCode,AnonID
0,489434,12,85048,4393
1,489434,12,79323,4393
2,489434,12,79323,4393
3,489434,48,22041,4393
4,489434,24,21232,4393


Using the raw quantities introduces bias from several sources. Some products are more expensive, so stores will buy fewer. This would mean the engine would prefer small cheap items that are often bought in bulk, even when they are less relevant. Also, large stores buy more items than small stores, so the engine would skew towards the preferences of the largest stores.

We could use a binary value representing purchased or not purchased, but that removes useful information. Buying something five times is a more significant indicator than buying it once. However, a raw "number of purchases" would skew towards older stores that have higher numbers for all products. Also, the difference between ordering once and twice is a more significant indicator than ordering 14 times compared to 15 times. To fit this, we can take the log of the purchase count.

The returns have to come out of that count, which is the whole reason the cancellation rows were kept. For each customer and product, n is the number of lines with a positive quantity minus the number of lines with a negative one, and the pair is kept only if that count and the net quantity are both positive. An order that was entirely returned then disappears instead of scoring as a purchase.

In [16]:
# set n = purchase count, net of returns
train_simple = train_simple.assign(is_buy=(train_simple['Quantity'] > 0).astype(int),
                                   is_ret=(train_simple['Quantity'] < 0).astype(int))

train_grouped = (train_simple.groupby(['AnonID', 'ProductCode'])
                 .agg(n=('is_buy', 'sum'), returns=('is_ret', 'sum'), net_qty=('Quantity', 'sum'))
                 .reset_index())
train_grouped['n'] = train_grouped['n'] - train_grouped['returns']

# a pair that was cancelled out by its returns is not a purchase
before = len(train_grouped)
train_grouped = train_grouped[(train_grouped['n'] > 0) & (train_grouped['net_qty'] > 0)]
print(f"Pairs cancelled out by returns: {before - len(train_grouped):,}")
print(f"Pairs remaining: {len(train_grouped):,}")

# log
train_grouped['n'] = np.log1p(train_grouped['n'])

Pairs cancelled out by returns: 6,885
Pairs remaining: 397,097


Products bought by fewer than 10 customers are dropped. A product bought once contributes an almost empty column but still costs a full factor vector in the model, and there is nothing for the decomposition to learn from it.

In [17]:
MIN_CUSTOMERS_PER_PRODUCT = 10

counts = train_grouped.groupby('ProductCode')['AnonID'].nunique()
keep = counts[counts >= MIN_CUSTOMERS_PER_PRODUCT].index

print(f"Products: {counts.size:,} -> {len(keep):,}")
train_grouped = train_grouped[train_grouped['ProductCode'].isin(keep)]
print(f"Cells: {len(train_grouped):,}")

Products: 3,762 -> 3,273


Cells: 394,764


The matrix is almost entirely zeros, so a dense table would waste nearly all of the memory it uses. `scipy.sparse` keeps only the non-zero cells. The two index maps are what let the ML engineer get from a row or column number back to a customer or a product, and what keeps the training matrix and the held-out pairs aligned.

In [18]:
# Reshape into a sparse customer x product matrix
customers = np.sort(train_grouped['AnonID'].unique())
products = np.sort(train_grouped['ProductCode'].unique())
cust_idx = {c: i for i, c in enumerate(customers)}
prod_idx = {p: j for j, p in enumerate(products)}

A = sp.csr_matrix(
    (train_grouped['n'].to_numpy(),
     (train_grouped['AnonID'].map(cust_idx).to_numpy(),
      train_grouped['ProductCode'].map(prod_idx).to_numpy())),
    shape=(len(customers), len(products)),
)

# Map the held-out pairs onto the same indices. A product that did not survive the
# filter has no factor vector, so it cannot be scored and is dropped.
pairs = pairs[pairs['AnonID'].isin(cust_idx) & pairs['ProductCode'].isin(prod_idx)]
holdout = pd.DataFrame({
    'customer_row': pairs['AnonID'].map(cust_idx),
    'product_col': pairs['ProductCode'].map(prod_idx),
}).sort_values(['customer_row', 'product_col']).reset_index(drop=True)

print(f"Held-out pairs scoreable against the matrix: {len(holdout):,}")
print(f"Test customers covered: {holdout['customer_row'].nunique():,}")
A

Held-out pairs scoreable against the matrix: 43,800
Test customers covered: 1,915


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 394764 stored elements and shape (5454, 3273)>

### An inner validation split, for choosing k

The ML Engineer has to choose k on something, and it cannot be the test set — tuning on the
held-out purchases and then reporting the winner's score would make that score meaningless.

A random sample of training purchases will not do either, and this is worth being precise about.
The test asks which products a shop buys **next**, in a period the model has never seen. A random
split asks which products it bought **at the same time** as the ones the model was given. Those are
different questions, and they prefer different models: the random version keeps rewarding more
components right up to k = 200, while the real one stops improving far earlier.

So the training period gets the same treatment the test period got — the last two months of it are
split off, and only products the customer had never bought before them are kept. Dates exist only
here, before the aggregation, which is why this split has to be built by the Data Engineer rather
than in the model notebook.

In [19]:
INNER_MONTHS = 2
inner_cutoff = cutoff - pd.DateOffset(months=INNER_MONTHS)

train_inner = train[train['InvoiceDate'] < inner_cutoff]
val_period = train[train['InvoiceDate'] >= inner_cutoff]

# Same R2 rule as the test hold-out: products this customer had never bought before
inner_seen = set(map(tuple, train_inner[['AnonID', 'ProductCode']].drop_duplicates().to_numpy()))
inner_customers = set(train_inner['AnonID'].unique())

vpairs = val_period[['AnonID', 'ProductCode']].drop_duplicates()
vpairs = vpairs[vpairs['AnonID'].isin(inner_customers)]
vpairs = vpairs[[tuple(p) not in inner_seen for p in vpairs.to_numpy()]]

# Same purchase value as the frozen matrix, on the inner training rows only
inner = train_inner.drop(columns=['Variation', 'InvoiceDate'])
inner = inner.assign(is_buy=(inner['Quantity'] > 0).astype(int),
                     is_ret=(inner['Quantity'] < 0).astype(int))
inner_grouped = (inner.groupby(['AnonID', 'ProductCode'])
                 .agg(n=('is_buy', 'sum'), returns=('is_ret', 'sum'), net_qty=('Quantity', 'sum'))
                 .reset_index())
inner_grouped['n'] = inner_grouped['n'] - inner_grouped['returns']
inner_grouped = inner_grouped[(inner_grouped['n'] > 0) & (inner_grouped['net_qty'] > 0)]
inner_grouped['n'] = np.log1p(inner_grouped['n'])

# Built on the SAME index as the frozen matrix, so rows and columns mean the same thing
inner_grouped = inner_grouped[inner_grouped['AnonID'].isin(cust_idx)
                              & inner_grouped['ProductCode'].isin(prod_idx)]
A_inner = sp.csr_matrix(
    (inner_grouped['n'].to_numpy(),
     (inner_grouped['AnonID'].map(cust_idx).to_numpy(),
      inner_grouped['ProductCode'].map(prod_idx).to_numpy())),
    shape=A.shape,
)

vpairs = vpairs[vpairs['AnonID'].isin(cust_idx) & vpairs['ProductCode'].isin(prod_idx)]
val_holdout = pd.DataFrame({
    'customer_row': vpairs['AnonID'].map(cust_idx),
    'product_col': vpairs['ProductCode'].map(prod_idx),
}).sort_values(['customer_row', 'product_col']).reset_index(drop=True)

print(f"Inner cutoff: {inner_cutoff:%Y-%m-%d}")
print(f"Inner training matrix: {A_inner.nnz:,} non-zeros (against {A.nnz:,} in the full one)")
print(f"Validation pairs: {len(val_holdout):,} across "
      f"{val_holdout['customer_row'].nunique():,} customers "
      f"({len(val_holdout) / val_holdout['customer_row'].nunique():.1f} per customer, against "
      f"{len(holdout) / holdout['customer_row'].nunique():.1f} in the test hold-out)")

Inner cutoff: 2011-08-09
Inner training matrix: 356,565 non-zeros (against 394,764 in the full one)
Validation pairs: 28,793 across 1,430 customers (20.1 per customer, against 22.9 in the test hold-out)


In [20]:
# Save the outputs
sp.save_npz(OUT / "train_matrix.npz", A)
pd.DataFrame({'row': np.arange(len(customers)), 'AnonID': customers}).to_csv(OUT / "customer_index.csv", index=False)
pd.DataFrame({'col': np.arange(len(products)), 'ProductCode': products}).to_csv(OUT / "product_index.csv", index=False)
holdout.to_csv(OUT / "test_holdout.csv", index=False)
sp.save_npz(OUT / "val_matrix.npz", A_inner)
val_holdout.to_csv(OUT / "val_holdout.csv", index=False)
product_names.reindex(products).rename('Description').to_csv(OUT / "product_names.csv")
pd.DataFrame({'row': np.arange(len(customers)),
              'Country': customer_country.reindex(customers).to_numpy()}).to_csv(OUT / "customer_country.csv", index=False)

for f in sorted(OUT.iterdir()):
    print(f"{f.name:<24} {f.stat().st_size / 1024:>10,.1f} KB")

customer_country.csv          102.3 KB
customer_index.csv             51.2 KB
product_index.csv              34.1 KB
product_names.csv             108.8 KB
test_holdout.csv              411.6 KB
train_matrix.npz              729.2 KB
val_holdout.csv               271.3 KB
val_matrix.npz                655.2 KB


## DE5 - Matrix report

Shape, non-zeros, sparsity, and the memory a dense table would have cost against what the
sparse one actually costs.

In [21]:
dense_mb = A.shape[0] * A.shape[1] * 8 / 1024**2
sparse_mb = (A.data.nbytes + A.indices.nbytes + A.indptr.nbytes) / 1024**2

print(f"Shape (customers x products): {A.shape[0]:,} x {A.shape[1]:,}")
print(f"Non-zero cells (nnz): {A.nnz:,}")
print(f"Sparsity: {1 - A.nnz / (A.shape[0] * A.shape[1]):.2%}")
print(f"Dense memory (float64): {dense_mb:,.1f} MB")
print(f"Sparse memory (CSR): {sparse_mb:,.1f} MB")
print(f"Compression: {dense_mb / sparse_mb:.0f}x smaller")
print(f"Held-out test pairs: {len(holdout):,}")
print(f"Train/test cutoff: {cutoff:%Y-%m-%d}")

Shape (customers x products): 5,454 x 3,273
Non-zero cells (nnz): 394,764
Sparsity: 97.79%
Dense memory (float64): 136.2 MB
Sparse memory (CSR): 4.5 MB
Compression: 30x smaller
Held-out test pairs: 43,800
Train/test cutoff: 2011-10-09
